# 11 — World 1 Full: Random Starts + Level Mixing

Trains a single PPO agent on **all four World 1 levels** (1-1, 1-2, 1-3, 1-4)
with **random starting positions** inside each level.

**Motivation:**
- Training on a single level from the start risks the agent memorising the
  fixed opening sequence instead of learning reusable skills.
- Random starts force exposure to diverse mid-level situations: pipe jumps,
  enemy patterns, gaps — whatever occurs anywhere in the level.
- Mixing all four World 1 stages in parallel further diversifies the gradient
  signal: underground (1-2), night (1-3), and castle (1-4) all share the
  same core mechanics but present them in different visual contexts.

**Setup:**
- Observation: flattened 13×16 symbolic grid × 4 frame-stack + power state = **833-dim** vector
  *(Task 1 appended Mario's normalised power state as the 833rd element)*
- Policy: `MlpPolicy` with `[512, 512]` hidden layers
- **8 parallel envs — 2 per World 1 level** (1-1, 1-2, 1-3, 1-4)
- **`random_start_steps=100`** — each reset advances Mario 0–100 steps into the level
- Higher entropy (`ent_coef=0.02`) to encourage exploration from unfamiliar positions
- Saves to `models/symbolic_ppo_world1_random/`

**Fine-tuning intent (Task 5):**
A pre-trained 1-1 model exists at `models/symbolic_ppo/final_model` (notebook 09).
However, that model was trained on a **(832,)** observation space, while this notebook
now uses **(833,)** after Task 1. Loading it directly raises a shape mismatch error.
To fine-tune, first retrain notebook 09 so it produces a **(833,)**-compatible base model,
then replace the `PPO(...)` cell below with `PPO.load('models/symbolic_ppo/final_model', env=env, ...)`.

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-Mario.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/lmartim4/Desktop/CSC-52081-Mario


In [2]:
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList

from src.wrappers import make_symbolic_multitask_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback, CurriculumCallback, PerLevelEvalCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using CUDA: True
GPU: NVIDIA GeForce RTX 3060 Ti


In [3]:
# All four World 1 levels — 2 parallel envs each = 8 total
# random_start_steps=100: at each reset Mario is advanced 0-100 steps
# (with skip=4 that is 0-400 raw frames, ~0-6s into the level)
LEVELS = [
    'SuperMarioBros-1-1-v3',
    'SuperMarioBros-1-2-v3',
    'SuperMarioBros-1-3-v3',
    'SuperMarioBros-1-4-v3',
]
ENVS_PER_LEVEL = 2  # equal weight across all four levels
RANDOM_START_STEPS = 100

env = make_symbolic_multitask_vec_env(
    env_ids=LEVELS,
    skip=4,
    n_stack=4,
    flatten=True,
    envs_per_level=ENVS_PER_LEVEL,
    random_start_steps=RANDOM_START_STEPS,
)

total_envs = ENVS_PER_LEVEL * len(LEVELS)
print(f'Levels: {LEVELS}')
print(f'Envs per level: {ENVS_PER_LEVEL}  |  Total envs: {total_envs}')
print(f'Random start: 0 - {RANDOM_START_STEPS} steps per episode')
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Levels: ['SuperMarioBros-1-1-v3', 'SuperMarioBros-1-2-v3', 'SuperMarioBros-1-3-v3', 'SuperMarioBros-1-4-v3']
Envs per level: 2  |  Total envs: 8
Random start: 0 - 100 steps per episode
Observation space: (833,)
Action space: 7


/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environm

In [ ]:
import glob
import re
import os

CHECKPOINT_DIR = 'models/symbolic_ppo_world1_random'

config = PPOConfig(
    policy='MlpPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.9,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02,   # higher entropy — agent starts from unfamiliar positions
    vf_coef=0.5,
    max_grad_norm=0.5,
)

# ---------------------------------------------------------------------------
# Auto-resume: find the most recently saved checkpoint to continue from
# ---------------------------------------------------------------------------
def find_latest_checkpoint(directory):
    """Return path of the most recently modified model, or None if none found."""
    if not os.path.isdir(directory):
        return None
    candidates = glob.glob(os.path.join(directory, 'model_*.zip'))
    final = os.path.join(directory, 'final_model.zip')
    if os.path.exists(final):
        candidates.append(final)
    if not candidates:
        return None
    return max(candidates, key=os.path.getmtime)

resume_path = find_latest_checkpoint(CHECKPOINT_DIR)

if resume_path:
    model = PPO.load(
        resume_path,
        env=env,
        device='cpu',
        learning_rate=config.learning_rate,
        ent_coef=config.ent_coef,
    )
    print(f'Resuming from checkpoint: {resume_path}')
    print(f'Steps already trained: {model.num_timesteps:,}')
else:
    model = PPO(
        policy=config.policy,
        env=env,
        learning_rate=config.learning_rate,
        n_steps=config.n_steps,
        batch_size=config.batch_size,
        n_epochs=config.n_epochs,
        gamma=config.gamma,
        gae_lambda=config.gae_lambda,
        clip_range=config.clip_range,
        ent_coef=config.ent_coef,
        vf_coef=config.vf_coef,
        max_grad_norm=config.max_grad_norm,
        policy_kwargs=dict(net_arch=[512, 512]),
        tensorboard_log='logs/symbolic_ppo_world1_random',
        verbose=1,
        device='cpu',
    )
    print(f'Training from scratch')

print(f'Device: {model.device}')
print(f'Batch per rollout: {config.n_steps * total_envs} samples')

In [5]:
%load_ext tensorboard
%tensorboard --logdir logs/symbolic_ppo_world1_random

Reusing TensorBoard on port 6009 (pid 40426), started 1:27:15 ago. (Use '!kill 40426' to kill it.)

In [ ]:
ADDITIONAL_STEPS = 4_000_000

# If resuming, extend the total beyond the already-trained steps
cumulative_total = model.num_timesteps + ADDITIONAL_STEPS

checkpoint_cb = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo_world1_random',
    save_freq=100_000,
)
curriculum_cb = CurriculumCallback(
    start_steps=0,
    end_steps=RANDOM_START_STEPS,
    total_timesteps=ADDITIONAL_STEPS,
)
eval_cb = PerLevelEvalCallback(
    levels=LEVELS,
    eval_freq=200_000,
    n_eval_episodes=5,
)

print(f'Already trained: {model.num_timesteps:,} steps')
print(f'Training for:    {ADDITIONAL_STEPS:,} more steps')
print(f'Cumulative total: {cumulative_total:,} steps')

model.learn(
    total_timesteps=cumulative_total,
    callback=CallbackList([checkpoint_cb, curriculum_cb, eval_cb]),
    log_interval=1,
    reset_num_timesteps=False,  # continue step counter — keeps TensorBoard curves continuous
)
model.save('models/symbolic_ppo_world1_random/final_model')
print('Training complete!')

## Evaluate on each level separately

In [7]:
import numpy as np
from stable_baselines3 import PPO

eval_model = PPO.load('models/symbolic_ppo_world1_random/final_model')

for level in LEVELS:
    print(f'\n{"="*50}')
    print(f'Evaluating on: {level}')
    print(f'{"="*50}')

    # Evaluate without random starts so results are deterministic & comparable
    eval_env = make_symbolic_env(
        env_id=level, skip=4, n_stack=4, flatten=True,
        random_start_steps=0,
    )

    rewards, flags = [], []
    
    result = eval_env.reset()
    obs = result[0] if isinstance(result, tuple) else result
    done, total_reward, flag = False, 0.0, False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        result = eval_env.step(int(action))
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result
        total_reward += float(reward)
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    flags.append(flag)
    status = 'FLAG!' if flag else 'DEAD'
    print(f'Reward={total_reward:.1f} [{status}]')

    print(f'\n  Mean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
    print(f'  Flag rate:   {np.mean(flags):.0%}')
    eval_env.close()

/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(



Evaluating on: SuperMarioBros-1-1-v3


/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


Reward=314.2 [FLAG!]

  Mean reward: 314.2 +/- 0.0
  Flag rate:   100%

Evaluating on: SuperMarioBros-1-2-v3
Reward=155.0 [DEAD]

  Mean reward: 155.0 +/- 0.0
  Flag rate:   0%

Evaluating on: SuperMarioBros-1-3-v3
Reward=66.4 [DEAD]

  Mean reward: 66.4 +/- 0.0
  Flag rate:   0%

Evaluating on: SuperMarioBros-1-4-v3
Reward=223.5 [FLAG!]

  Mean reward: 223.5 +/- 0.0
  Flag rate:   100%


## Evaluate on each level with random starts

In [8]:
import numpy as np
from stable_baselines3 import PPO

eval_model = PPO.load('models/symbolic_ppo_world1_random/final_model')

EVAL_RANDOM_START_STEPS = 100  # same range used during training
N_EPISODES = 10

for level in LEVELS:
    print(f'\n{"="*50}')
    print(f'Evaluating on: {level}  (random start 0–{EVAL_RANDOM_START_STEPS} steps)')
    print(f'{"="*50}')

    eval_env = make_symbolic_env(
        env_id=level, skip=4, n_stack=4, flatten=True,
        random_start_steps=EVAL_RANDOM_START_STEPS,
    )

    rewards, flags = [], []
    for ep in range(N_EPISODES):
        result = eval_env.reset()
        obs = result[0] if isinstance(result, tuple) else result
        done, total_reward, flag = False, 0.0, False

        while not done:
            action, _ = eval_model.predict(obs, deterministic=True)
            result = eval_env.step(int(action))
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            total_reward += float(reward)
            if isinstance(info, dict) and info.get('flag_get', False):
                flag = True

        rewards.append(total_reward)
        flags.append(flag)
        status = 'FLAG!' if flag else 'DEAD'
        print(f'  Ep {ep+1:2d}: reward={total_reward:.1f} [{status}]')

    print(f'\n  Mean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
    print(f'  Flag rate:   {np.mean(flags):.0%}')
    eval_env.close()


Evaluating on: SuperMarioBros-1-1-v3  (random start 0–100 steps)


/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


  Ep  1: reward=309.5 [FLAG!]
  Ep  2: reward=260.5 [FLAG!]
  Ep  3: reward=287.4 [FLAG!]
  Ep  4: reward=312.3 [FLAG!]
  Ep  5: reward=279.1 [FLAG!]
  Ep  6: reward=270.2 [FLAG!]
  Ep  7: reward=293.9 [FLAG!]
  Ep  8: reward=134.1 [DEAD]
  Ep  9: reward=310.9 [FLAG!]
  Ep 10: reward=314.2 [FLAG!]

  Mean reward: 277.2 +/- 51.0
  Flag rate:   90%

Evaluating on: SuperMarioBros-1-2-v3  (random start 0–100 steps)
  Ep  1: reward=155.0 [DEAD]
  Ep  2: reward=81.6 [DEAD]
  Ep  3: reward=155.2 [DEAD]
  Ep  4: reward=65.8 [DEAD]
  Ep  5: reward=155.0 [DEAD]
  Ep  6: reward=-3.9 [DEAD]
  Ep  7: reward=154.1 [DEAD]
  Ep  8: reward=35.1 [DEAD]
  Ep  9: reward=155.0 [DEAD]
  Ep 10: reward=54.5 [DEAD]

  Mean reward: 100.8 +/- 58.0
  Flag rate:   0%

Evaluating on: SuperMarioBros-1-3-v3  (random start 0–100 steps)
  Ep  1: reward=1.1 [DEAD]
  Ep  2: reward=62.2 [DEAD]
  Ep  3: reward=66.4 [DEAD]
  Ep  4: reward=-5.2 [DEAD]
  Ep  5: reward=46.3 [DEAD]
  Ep  6: reward=66.4 [DEAD]
  Ep  7: reward=62

In [ ]:
# Watch any level:
# python watch_agent.py --model models/symbolic_ppo_world1_random/final_model --env SuperMarioBros-1-1-v3
# python watch_agent.py --model models/symbolic_ppo_world1_random/final_model --env SuperMarioBros-1-2-v3
# python watch_agent.py --model models/symbolic_ppo_world1_random/final_model --env SuperMarioBros-1-3-v3
# python watch_agent.py --model models/symbolic_ppo_world1_random/final_model --env SuperMarioBros-1-4-v3